# Qwen Studio — Heretic GGUF v8.6

Interface **Gradio premium**, mobile-first.

- Qwen-Image 2.1
- Heretic GGUF Q4_K_M
- édition + face swap image
- **face swap vidéo Ultra : analyse, tracking, bouche, pause/reprise, progression, annulation**
- jobs persistants
- cache Google Drive
- images et vidéos affichées **uniquement dans Gradio**


In [ ]:
#@title ▶ Lancer Qwen Studio v8.6
import urllib.request, urllib.error, json, os, time

REPO = "dbrckk/Colab"
QWEN_STUDIO_VERSION = "8.6"
LOCAL = "/content/qwen_studio_v8"
os.makedirs(LOCAL, exist_ok=True)

# Épingler tous les modules au même commit GitHub évite de charger un mélange
# de versions si le repo est modifié pendant le démarrage du notebook.
try:
    req = urllib.request.Request(
        f"https://api.github.com/repos/{REPO}/commits/main",
        headers={"Accept": "application/vnd.github+json", "User-Agent": "QwenStudio-Colab"},
    )
    with urllib.request.urlopen(req, timeout=20) as response:
        QWEN_STUDIO_SOURCE_REF = json.load(response)["sha"]
except Exception:
    QWEN_STUDIO_SOURCE_REF = "main"

BASE = f"https://raw.githubusercontent.com/{REPO}/{QWEN_STUDIO_SOURCE_REF}/qwen_studio_v8"

GROUPS = [
    ["01_setup.py"],
    ["02_models.py"],
    ["03_runtime.py"],
    ["04_faceswap.py"],
    ["05_backend.part00", "05_backend.part01", "05_backend.part02"],
    ["05b_video_faceswap.py"],
    ["06_ui.part00", "06_ui.part01", "06_ui.part02", "06_ui.part03"],
]

def fetch_text(name, retries=3):
    url = f"{BASE}/{name}"
    last = None
    for attempt in range(1, retries + 1):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "QwenStudio-Colab"})
            with urllib.request.urlopen(req, timeout=60) as response:
                data = response.read()
            if len(data) < 8:
                raise RuntimeError(f"Fichier vide ou incomplet: {name}")
            return data.decode("utf-8")
        except Exception as exc:
            last = exc
            if attempt < retries:
                time.sleep(attempt)
    raise RuntimeError(f"Téléchargement impossible après {retries} essais: {name}: {last}")

print(f"Qwen Studio v{QWEN_STUDIO_VERSION} • source {QWEN_STUDIO_SOURCE_REF[:12]}")

for index, group in enumerate(GROUPS, 1):
    source = "".join(fetch_text(name) for name in group)
    filename = f"qwen_studio_v8_{index}.py"
    local_path = os.path.join(LOCAL, filename)
    # Compiler avant écriture/exécution : une erreur de téléchargement ou de
    # syntaxe ne lance jamais un module partiel.
    compiled = compile(source, local_path, "exec")
    tmp = local_path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as fh:
        fh.write(source)
    os.replace(tmp, local_path)
    print(f"✓ Étape {index}/{len(GROUPS)}")
    exec(compiled, globals())
